In [3]:
import pandas as pd

# Path to your downloaded .txt file
input_file = '/Users/balajisk/Developer/Masters/data_mining_&_data_analytical/data_mining_ber/irish-home-retrofit-prediction/45_Col_final_cleaned_data.csv'
# Path where you want to save the new .csv
output_file = './BERPublicsearch.csv'


chunk_size = 100000
null_counts = None
total_rows = 0

for chunk in pd.read_csv(input_file, chunksize=chunk_size, low_memory=False, encoding='latin-1'):
    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()
    total_rows += len(chunk)

# Get the list
null_percentages = null_counts / total_rows
high_null_cols = null_percentages[null_percentages > 0.2].index.tolist()

print(f"Total Rows: {total_rows}")
print("Columns to remove (>50% null):", high_null_cols)

Total Rows: 1351582
Columns to remove (>50% null): []


In [7]:
sorted_null_counts = null_counts.sort_values(ascending=False)

In [11]:
print(sorted_null_counts[sorted_null_counts>0])


PredominantRoofType          141260
NoOfSidesSheltered            24127
PercentageDraughtStripped     24127
SuspendedWoodenFloor          24127
StructureType                 24127
VentilationMethod             24127
NoOfFansAndVents              24127
HSSupplHeatFraction           23467
WHRenewableResources          23467
SupplWHFuel                   23467
SupplSHFuel                   23467
WHEffAdjFactor                23467
WHMainSystemEff               23467
HSSupplSystemEff              23467
SHRenewableResources          23467
HSMainSystemEfficiency        23467
HSEffAdjFactor                23467
dtype: int64


In [13]:
import pandas as pd

# 1. Define your file paths
cleaned_csv_path = '45_Col_final_cleaned_data.csv'
# Replace this with the actual path to your "all data" file (e.g., the original raw data)
source_file_path = 'BER_Cleaned_Data.parquet' 
output_parquet_path = '46_Col_final_with_county.parquet'

# 2. Get the list of 45 column names from your cleaned CSV
temp_df = pd.read_csv(cleaned_csv_path, nrows=0) # Reads only the header
columns_45 = temp_df.columns.tolist()

# 3. Add 'CountyName' to the list to make 46 columns
# (Ensure 'CountyName' matches the exact name in your source file)
columns_46 = ['CountyName'] + columns_45

print(f"Selecting {len(columns_46)} columns...")

# 4. Load only the 46 columns from the source file
# If the source is a Parquet file:
df_final = pd.read_parquet(source_file_path, columns=columns_46)

# Alternatively, if your source is a CSV/TXT, use this:
# df_final = pd.read_csv(source_file_path, usecols=columns_46, low_memory=False, encoding='latin-1')

# 5. Save the result to a new Parquet file
df_final.to_parquet(output_parquet_path, index=False)

print(f"Success! New file created: {output_parquet_path}")
print(f"New Shape: {df_final.shape}")


Selecting 46 columns...
Success! New file created: 46_Col_final_with_county.parquet
New Shape: (1351582, 46)


In [15]:
import pandas as pd
import os

# Define paths
input_file = '46_Col_final_with_county.parquet'
clean_file = 'complete_data.csv'
null_file = 'rows_with_nulls.csv'

# Delete existing files if you are re-running this
for f in [clean_file, null_file]:
    if os.path.exists(f):
        os.remove(f)

chunk_size = 100000
total_clean = 0
total_nulls = 0

print("Starting split...")

# Process the file in chunks
for i, chunk in enumerate(pd.read_parquet(input_file, chunksize=chunk_size, low_memory=False, encoding='latin-1')):
    # 1. Rows with NO null values
    clean_chunk = chunk.dropna()
    
    # 2. Rows with ANY null values (find rows where any column is null)
    null_chunk = chunk[chunk.isnull().any(axis=1)]
    
    # Save clean rows (write header only for the first chunk)
    clean_chunk.to_csv(clean_file, mode='a', index=False, header=(not os.path.exists(clean_file)))
    
    # Save null rows (write header only for the first chunk)
    null_chunk.to_csv(null_file, mode='a', index=False, header=(not os.path.exists(null_file)))
    
    total_clean += len(clean_chunk)
    total_nulls += len(null_chunk)
    
    print(f"Processed chunk {i+1} | Complete: {total_clean} | Missing: {total_nulls}")

print("\nSplit Complete!")
print(f"Saved complete rows to: {clean_file}")
print(f"Saved rows with nulls to: {null_file}")


Starting split...


TypeError: read_table() got an unexpected keyword argument 'chunksize'

In [16]:
import pandas as pd
import pyarrow.parquet as pq

# Create a Parquet file reader
parquet_file = pq.ParquetFile(input_file)

# Iterate through "Row Groups" (the Parquet version of chunks)
for i in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(i).to_pandas()
    
    # Process your chunk here...
    clean_chunk = chunk.dropna()
    null_chunk = chunk[chunk.isnull().any(axis=1)]
    
    # Append to your files
    clean_chunk.to_csv('complete_data.csv', mode='a', index=False, header=(i==0))
    null_chunk.to_csv('rows_with_nulls.csv', mode='a', index=False, header=(i==0))
    
    print(f"Processed row group {i+1}")


Processed row group 1
Processed row group 2
